# Exploratory Data Analysis (EDA) on Customer Purchase Dataset

This notebook performs a complete **Exploratory Data Analysis (EDA)** on a customer purchase dataset. 
The goal of EDA is to understand the structure, quality, and patterns in the data **before** building 
any Machine Learning model.

**What we will do:**
1. Load and inspect the dataset
2. Handle missing values
3. Generate descriptive statistics
4. Visualize the data using:
   - Histograms
   - Scatter plots
   - Box plots
   - Correlation heatmap
   - Pair plots
5. Summarize insights

**Dataset columns:** `Customer_ID`, `Age`, `Gender`, `City`, `Salary`, `Purchase_Amount`, `Is_Member`, `Purchase_Date`


## 1. Import Required Libraries

We import `pandas` and `numpy` for data handling, and `matplotlib` / `seaborn` for visualization.
`seaborn.set()` gives our plots a clean, consistent style.


In [ ]:
# Data handling libraries
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean visual theme for all seaborn plots
sns.set(style="whitegrid")

# Ensures plots render directly inside the notebook (needed in Jupyter, harmless in Colab)
%matplotlib inline


## 2. Load the Dataset

We load the CSV file into a pandas DataFrame called `df`.
If you're running this in **Google Colab**, upload the CSV first using the Files pane,
or use `from google.colab import files` + `files.upload()`.


In [ ]:
# Load the dataset from a CSV file into a pandas DataFrame
df = pd.read_csv("EDA_Practice_Dataset.csv")

# Display the first 5 rows to get a quick look at the data
df.head()


## 3. Descriptive Statistics

Before visualizing anything, we check:
- The **shape** of the dataset (rows x columns)
- **Data types** of each column
- **Missing values** in each column
- **Summary statistics** (mean, std, min, max, quartiles) for numeric columns
- **Summary statistics** for categorical (text) columns

This step helps us understand what cleaning might be needed before analysis.


In [ ]:
# Number of rows and columns in the dataset
print("Shape of dataset (rows, columns):", df.shape)

# Data types of each column + count of non-null values
print("\n--- Data types and non-null counts ---")
df.info()

# Count of missing (NaN) values per column
print("\n--- Missing values per column ---")
print(df.isnull().sum())

# Statistical summary for numerical columns: count, mean, std, min, 25%, 50%, 75%, max
print("\n--- Statistical summary (numerical columns) ---")
print(df.describe())

# Statistical summary for categorical/text columns: count, unique, top, freq
print("\n--- Statistical summary (categorical columns) ---")
print(df.describe(include='object'))


## 4. Handling Missing Values

The dataset contains missing values in `Gender`, `Is_Member`, `Purchase_Date`, and possibly other columns.
For the purpose of this EDA:
- We keep the **original dataset** (`df`) untouched for reference.
- We create a **cleaned copy** (`df_clean`) where:
  - Missing numerical values are filled with the **median** (robust to outliers).
  - Missing categorical values are filled with the **mode** (most frequent value).

We use `df_clean` for plots that cannot handle missing values (like pair plots).


In [ ]:
# Create a copy of the original dataframe so we don't lose the raw data
df_clean = df.copy()

# Fill missing numerical values with the median of that column
numerical_cols = ['Age', 'Salary', 'Purchase_Amount']
for col in numerical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Fill missing categorical values with the mode (most frequent value) of that column
categorical_cols = ['Gender', 'City', 'Is_Member']
for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# Confirm there are no more missing values in the cleaned dataset (except Purchase_Date, which we don't use for plots)
print("Missing values after cleaning:")
print(df_clean.isnull().sum())


## 5. Univariate Analysis — Histograms

A **histogram** shows the distribution of a single numerical variable — how values are spread out,
whether the data is skewed, and whether there are unusual spikes or gaps.

We plot histograms for `Age`, `Salary`, and `Purchase_Amount`.


In [ ]:
# Plot histograms for all three numerical columns at once
df_clean[['Age', 'Salary', 'Purchase_Amount']].hist(
    figsize=(10, 6),   # overall figure size
    bins=15,           # number of bars/buckets
    color='skyblue',
    edgecolor='black'
)
plt.suptitle("Histograms of Numerical Features", fontsize=14)
plt.tight_layout()  # prevents titles/labels from overlapping
plt.show()


In [ ]:
# A closer look at Salary distribution, with a KDE (Kernel Density Estimate) curve
# The KDE line smooths the histogram to show the overall shape of the distribution
plt.figure(figsize=(6, 4))
sns.histplot(df_clean['Salary'], kde=True, color='teal')
plt.title("Distribution of Salary")
plt.xlabel("Salary")
plt.ylabel("Frequency")
plt.show()


## 6. Bivariate Analysis — Scatter Plot

A **scatter plot** shows the relationship between two numerical variables.
Here we plot `Salary` vs `Purchase_Amount`, colored by `Gender`, to see if higher salary
tends to relate to higher purchase amounts, and whether this differs by gender.


In [ ]:
plt.figure(figsize=(6, 4))

# x and y are the two numerical variables being compared
# hue splits the points by color based on the Gender column
sns.scatterplot(data=df_clean, x='Salary', y='Purchase_Amount', hue='Gender')

plt.title("Salary vs Purchase Amount")
plt.xlabel("Salary")
plt.ylabel("Purchase Amount")
plt.show()


## 7. Box Plots

A **box plot** shows the median, quartiles, and potential outliers of a numerical variable.
- The box represents the interquartile range (IQR) — the middle 50% of the data.
- The line inside the box is the median.
- Points outside the "whiskers" are potential outliers.

We compare `Purchase_Amount` across different cities, and separately check `Salary` for outliers.


In [ ]:
# Box plot comparing Purchase_Amount across different cities
plt.figure(figsize=(8, 4))
sns.boxplot(data=df_clean, x='City', y='Purchase_Amount', palette='Set2')
plt.title("Purchase Amount by City")
plt.xlabel("City")
plt.ylabel("Purchase Amount")
plt.xticks(rotation=45)  # rotate x-axis labels so they don't overlap
plt.show()


In [ ]:
# Box plot to check for outliers in Salary
plt.figure(figsize=(4, 5))
sns.boxplot(y=df_clean['Salary'], color='lightcoral')
plt.title("Boxplot of Salary")
plt.show()


## 8. Correlation Heatmap

A **correlation heatmap** shows how strongly numerical variables are related to each other.
- Values range from **-1 to 1**.
- Close to **+1** → strong positive correlation (both increase together)
- Close to **-1** → strong negative correlation (one increases as the other decreases)
- Close to **0** → little to no linear relationship


In [ ]:
# Compute the correlation matrix for numerical columns only
correlation_matrix = df_clean[['Age', 'Salary', 'Purchase_Amount']].corr()

plt.figure(figsize=(5, 4))
sns.heatmap(
    correlation_matrix,
    annot=True,      # show the correlation values on the heatmap
    cmap='coolwarm', # color scheme: blue = negative, red = positive
    fmt=".2f",       # round values to 2 decimal places
    linewidths=0.5
)
plt.title("Correlation Heatmap")
plt.show()


## 9. Pair Plots

A **pair plot** shows scatter plots for every pair of numerical variables at once,
plus a distribution plot for each variable on the diagonal. This gives a quick overview
of all relationships in the dataset in a single figure.


In [ ]:
# Pair plot of Age, Salary, and Purchase_Amount, colored by Gender
# diag_kind='kde' shows a smooth density curve on the diagonal instead of a histogram
sns.pairplot(
    df_clean[['Age', 'Salary', 'Purchase_Amount', 'Gender']],
    hue='Gender',
    diag_kind='kde',
    palette='husl'
)
plt.suptitle("Pair Plot of Dataset", y=1.02)
plt.show()


## 10. Conclusion & Insights

Based on the EDA performed above:

- **Missing data:** The original dataset had missing values in `Gender`, `Is_Member`, `Purchase_Date`, 
  and some numerical columns. These were imputed (median for numeric, mode for categorical) in `df_clean`.
- **Distributions:** *(Fill in what you observe — e.g., "Salary is right-skewed, with most customers 
  earning between X and Y, and a few high earners pulling the distribution to the right.")*
- **Salary vs Purchase Amount:** *(Fill in — e.g., "There appears to be a weak/moderate positive 
  relationship between Salary and Purchase Amount.")*
- **City-wise spending:** *(Fill in — e.g., "Customers in [City] show higher median purchase amounts, 
  with a few outliers.")*
- **Correlation:** *(Fill in — e.g., "Salary and Purchase_Amount show a correlation of ~0.XX, indicating 
  a weak/moderate/strong relationship.")*

> **Note:** Replace the italicized placeholders above with the actual patterns you observe in your 
> own plot outputs — instructors typically look for genuine interpretation, not just working code.

These insights help guide **feature selection** and **preprocessing** decisions for the next stage 
of the Machine Learning pipeline.
